# 03 – Preprocessing & Feature Engineering

Baut Lags, Rolling-Fenster und Kalender-Features. **Train + Test werden zusammengefügt**, damit Lags am Test-Anfang die Train-Historie nutzen.

| Modus | Wo | Daten | RAM |
|-------|-----|--------|-----|
| `sample` | Lokal | `train_sample.csv` + `test.csv` | ~500 MB |
| `full` | **Colab (CPU)** | voller `train.csv` + `test.csv` | ~6–10 GB empfohlen |

**Outputs** (für Notebook 04):
- `outputs/processed/train_labeled.parquet` – nur Zeilen mit `score`
- `outputs/processed/test_features.parquet` – alle Test-Zeilen

Siehe [docs/06_EDA_ANALYSIS.md](../docs/06_EDA_ANALYSIS.md) für die Feature-Logik.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists() and (PROJECT_ROOT.parent / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from scripts.features import build_features, feature_columns, add_persistence_baseline

print("PROJECT_ROOT (vor Setup):", PROJECT_ROOT)

## 1. Umgebung & Modus

In [ ]:
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    if IS_COLAB:
        candidates += [
            Path("/content/DataMining_Final-Project"),
            Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project"),
        ]
    for p in candidates:
        if (p / "config" / "paths.py").exists():
            return p.resolve()
    return Path.cwd().resolve()


if IS_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

try:
    from config.paths import setup_environment
    env = setup_environment()
except ModuleNotFoundError:
    data_dir = Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project/data")
    if not data_dir.exists():
        data_dir = PROJECT_ROOT / "data"
    env = {
        "is_colab": IS_COLAB,
        "data_dir": data_dir,
        "train_path": data_dir / "train.csv",
        "test_path": data_dir / "test.csv",
    }

DATA_DIR = Path(env["data_dir"])
TRAIN_PATH = Path(env["train_path"])
TEST_PATH = Path(env["test_path"])

# sample = lokal testen | full = Colab mit vollem Train
MODE = "full" if IS_COLAB else "sample"
# MODE = "sample"  # zum manuellen Überschreiben

if MODE == "sample":
    TRAIN_PATH = DATA_DIR / "train_sample.csv"
    if not TRAIN_PATH.exists():
        raise FileNotFoundError("train_sample.csv fehlt — python scripts/create_sample.py")
else:
    TRAIN_PATH = DATA_DIR / "train.csv"

print(f"Umgebung: {'Colab' if IS_COLAB else 'Lokal'} | Modus: {MODE}")
print(f"  TRAIN: {TRAIN_PATH}")
print(f"  TEST:  {TEST_PATH}")
for p, name in [(TRAIN_PATH, "Train"), (TEST_PATH, "Test")]:
    print(f"  {'✓' if p.exists() else '✗'} {name}: {p}")

PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Panel laden (Train + Test)

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train["_split"] = "train"
test["_split"] = "test"
test["score"] = np.nan

panel = pd.concat([train, test], ignore_index=True)
print(f"Panel: {len(panel):,} Zeilen | Train {len(train):,} | Test {len(test):,}")
print(f"Regionen: {panel['region_id'].nunique()} | Spalten: {panel.shape[1]}")

## 3. Features berechnen

Dauert im `full`-Modus in Colab typisch **5–20 Min.** (CPU, ein Durchlauf `groupby` pro Region).

In [ ]:
panel = build_features(panel)
panel = add_persistence_baseline(panel, lag_days=7)

feat_cols = feature_columns()
print(f"Feature-Spalten: {len(feat_cols)}")
print(f"Panel nach Features: {panel.shape}")
panel.head(3)

## 4. Trennen & speichern

In [ ]:
train_feat = panel[panel["_split"] == "train"].copy()
test_feat = panel[panel["_split"] == "test"].copy()

train_labeled = train_feat[train_feat["score"].notna()].copy()

meta_cols = ["region_id", "date", "year", "month", "day", "ordinal", "score", "score_persist7"]
save_train_cols = list(dict.fromkeys(meta_cols + feat_cols))
save_train_cols = [c for c in save_train_cols if c in train_labeled.columns]
save_test_cols = list(dict.fromkeys(
    ["region_id", "date", "year", "month", "day", "ordinal"] + feat_cols
))
save_test_cols = [c for c in save_test_cols if c in test_feat.columns]

path_train = PROCESSED_DIR / f"train_labeled_{MODE}.parquet"
path_test = PROCESSED_DIR / f"test_features_{MODE}.parquet"

train_labeled[save_train_cols].to_parquet(path_train, index=False)
test_feat[save_test_cols].to_parquet(path_test, index=False)

# Symlinks für Notebook 04 (Standardnamen)
path_train_std = PROCESSED_DIR / "train_labeled.parquet"
path_test_std = PROCESSED_DIR / "test_features.parquet"
train_labeled[save_train_cols].to_parquet(path_train_std, index=False)
test_feat[save_test_cols].to_parquet(path_test_std, index=False)

print(f"Gespeichert:")
print(f"  {path_train_std}  ({len(train_labeled):,} Zeilen, labeled)")
print(f"  {path_test_std}   ({len(test_feat):,} Zeilen)")
print(f"  Label-Rate: {train_labeled['score'].notna().mean():.2%} (sollte 100% sein)")

## 5. Sanity-Check

In [ ]:
missing_feat = train_labeled[feat_cols].isna().mean().sort_values(ascending=False).head(10)
print("Top fehlende Features (wegen Lags am Serienanfang):")
display(missing_feat)

print("\nScore-Verteilung (labeled train):")
print(train_labeled["score"].describe())

if MODE == "full":
    print(f"\nRegionen: {train_labeled['region_id'].nunique()}")
    print(f"Zeilen/Region (labeled): {len(train_labeled) / train_labeled['region_id'].nunique():.0f}")